# Predictor template

1. `encode_batch` reads sequences and builds the model input.
2. `dummy_model` turns that input into coordinates. Replace this function when a real model is ready.
3. `write_pdb` writes one CA atom per residue.

In [2]:
!pip install torch

  Using cached torch-2.15.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (36 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.2.2-py2.py3-none-any.whl.metadata (20 kB)
  Using cached nvidia_cudnn_cu13-9.26.0.51-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nvshmem_cu13-3.7.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.2 kB)
  Using cached triton-3.9.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (978 bytes)
  Using cached nv

In [1]:
import os

# This WSL driver crashes inside the CUDA 13 runtime check. Ask PyTorch to use NVML instead.
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"

import numpy as np
import torch
from pathlib import Path
from skbio.io import read

ModuleNotFoundError: No module named 'torch'

In [ ]:
DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 20 amino acids, X for anything else, PAD for positions added to reach MAX_LENGTH
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY") + ["X", "PAD"]
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
MAX_LENGTH = 1024
BATCH_SIZE = 16

THREE_LETTER = {
    "A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU", "F": "PHE",
    "G": "GLY", "H": "HIS", "I": "ILE", "K": "LYS", "L": "LEU",
    "M": "MET", "N": "ASN", "P": "PRO", "Q": "GLN", "R": "ARG",
    "S": "SER", "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR",
    "X": "UNK",
}


## 1. Input

`read_fasta` uses `skbio.io.read`. `encode_batch` keeps the first `max_length` residues, pads the rest, and returns:

- `tokens` with shape `(B, D, N)` — one-hot, `N = 22`
- `mask` with shape `(B, D)` — `True` for a real residue, `False` for padding
- the letter strings that were actually encoded


In [ ]:
def read_fasta(path):
    records = []
    for seq in read(str(path), format="fasta"):
        seq_id = seq.metadata.get("id") or f"sequence_{len(records) + 1}"
        records.append((str(seq_id), str(seq)))
    return records


def encode_batch(records, max_length=MAX_LENGTH):
    standard = set("ACDEFGHIKLMNPQRSTVWY")
    n_aa = len(AMINO_ACIDS)
    tokens = torch.zeros(len(records), max_length, n_aa)
    mask = torch.zeros(len(records), max_length, dtype=torch.bool)
    sequences = []

    for i, (seq_id, sequence) in enumerate(records):
        letters = [aa if aa in standard else "X" for aa in sequence.upper()]
        letters = letters[:max_length]
        sequences.append("".join(letters))

        for j, aa in enumerate(letters):
            tokens[i, j, AA_TO_IDX[aa]] = 1
            mask[i, j] = True
        for j in range(len(letters), max_length):
            tokens[i, j, AA_TO_IDX["PAD"]] = 1

    return tokens, mask, sequences


## 2. Model

Input is the tensor from the encoder. Output is one `(L, 3)` array per protein: X, Y, Z of each CA atom. `L` is the number of `True` values in that row of the mask, so padding is not returned.

This version draws random numbers and ignores the amino-acid channels. A later model has to accept the same two arguments and return the same kind of list.


In [ ]:
def dummy_model(tokens, mask):
    batch_size, length, n_aa = tokens.shape
    coords = torch.randn(batch_size, length, 3) * 10

    per_protein = []
    for i in range(batch_size):
        per_protein.append(coords[i][mask[i]])
    return per_protein


## 3. Output

One PDB file per protein, one `ATOM` line per residue. The line is the wwPDB ATOM layout: atom name ` CA ` in columns 13–16, coordinates in columns 31–54, element `C` in columns 77–78.


In [ ]:
def write_pdb(sequence, coords, path):
    coords = np.asarray(coords)
    lines = []
    for i, (aa, xyz) in enumerate(zip(sequence, coords), start=1):
        res_name = THREE_LETTER.get(aa, "UNK")
        line = (
            f"ATOM  {i:5d}  CA  {res_name} A{i:4d}    "
            f"{xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}  1.00  0.00           C  "
        )
        lines.append(line)
    lines.append("END")
    path.write_text("\n".join(lines) + "\n")


## Run

Change `fasta_path` to any multi-protein FASTA. To swap a stage, replace `encode_batch`, `dummy_model`, or `write_pdb` in the loop below and leave the other two as they are.


In [ ]:
fasta_path = DATA_DIR / "examples" / "mini_proteins.fasta"
fasta_path.parent.mkdir(parents=True, exist_ok=True)
fasta_path.write_text(
    """>short
MKT
>odd_letters B, Z, O and U are not standard amino acids
MKTXBZOU
>wrapped
ACDEF
GHIKL
MNPQR
STVWY
ACDE
"""
)

records = read_fasta(fasta_path)
print(f"{len(records)} sequences:", [seq_id for seq_id, _ in records])

torch.manual_seed(0)
written = []
for start in range(0, len(records), BATCH_SIZE):
    batch = records[start : start + BATCH_SIZE]
    tokens, mask, sequences = encode_batch(batch)
    print("tokens", tuple(tokens.shape), "lengths", [len(s) for s in sequences])

    predictions = dummy_model(tokens, mask)
    for (seq_id, _), sequence, coords in zip(batch, sequences, predictions):
        path = OUT_DIR / f"{seq_id}.pdb"
        write_pdb(sequence, coords.numpy(), path)
        written.append(path)

written


In [ ]:
tokens, mask, sequences = encode_batch(records)
print("full file", tuple(tokens.shape), sequences)

# longer than D is cut; padding is not written
long_tokens, long_mask, long_seq = encode_batch([("long", "A" * 1100)], max_length=8)
print("truncated", long_seq, "mask sum", int(long_mask.sum()), "coords", tuple(dummy_model(long_tokens, long_mask)[0].shape))

for path, sequence in zip(written, sequences):
    atoms = [line for line in path.read_text().splitlines() if line.startswith("ATOM")]
    assert len(atoms) == len(sequence)
    assert len(atoms[0]) == 80
    assert atoms[0][12:16] == " CA "
    assert atoms[0][76:78] == " C"

print(written[0].name)
print(written[0].read_text())
